## 1. Setup

`statsmodels` supplies the OLS benchmark, and TensorFlow Keras supplies the network that is checked against it.

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from sklearn.metrics import r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.utils import plot_model

pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

## 2. The data

In [ ]:
panel = pd.read_csv("FINANCE384_assignmentA_development_panel.csv", parse_dates=["date"])

print(f"Rows           : {len(panel):,}")
print(f"Distinct stocks: {panel['permno'].nunique():,}")
print(f"Months         : {panel['date'].nunique():,} "
      f"({panel['date'].min():%Y-%m} to {panel['date'].max():%Y-%m})")
panel.head()

# Check 1: is (permno, date) a unique key? If not, every groupby and merge below is unreliable.
print(f"Duplicate (permno, date) rows: {panel.duplicated(['permno', 'date']).sum()}")

# Check 2: how many stocks are in each month, and how many months does a stock survive?
stocks_per_month = panel.groupby("date")["permno"].nunique()
months_per_stock = panel.groupby("permno")["date"].nunique()
print(f"Stocks per month : {stocks_per_month.min()} to {stocks_per_month.max()} "
      f"(mean {stocks_per_month.mean():.0f})")
print(f"Months per stock : {months_per_stock.min()} to {months_per_stock.max()} "
      f"(median {months_per_stock.median():.0f})   -> unbalanced")

# Check 3: missing values in the columns we model with
print("\nMissing values:")
print(panel[FEATURES + ["ret_excess_t"]].isna().sum())

# Check 4: do the feature distributions look reasonable?
print("\nFeature summary:")
panel[FEATURES + ["ret_excess_t"]].describe().loc[["mean", "std", "min", "max"]]

Rows           : 198,298
Distinct stocks: 1,252
Months         : 396 (1990-01 to 2022-12)


,date,permno,ticker,comnam,industry,ff49_code,ff49_name,gvkey,size,bm,...,divyield,gross_profit,roe,asset_growth,leverage,accruals,mkt_12m,mkt_vol_12m,down_market,ret_excess_t
0,1990-01-31,10057,AMT,ACME CLEVELAND CORP,Manufacturing,21,Mach,1098.0,17.9819,0.9481,...,0.0390,0.3981,-0.2042,-0.2392,0.1135,-0.2715,0.0347,0.0391,0,-0.0540
1,1990-02-28,10057,AMT,ACME CLEVELAND CORP,Manufacturing,21,Mach,1098.0,17.9696,0.9598,...,0.0395,0.3981,-0.2044,-0.2392,0.1135,-0.2318,0.0701,0.0383,0,-0.0179
2,1990-03-30,10057,AMT,ACME CLEVELAND CORP,Manufacturing,21,Mach,1098.0,17.9190,1.0772,...,0.0416,0.4351,0.0843,0.0563,0.0977,-0.0678,0.0729,0.0383,0,-0.0558
3,1990-04-30,10057,AMT,ACME CLEVELAND CORP,Manufacturing,21,Mach,1098.0,17.8656,1.1363,...,0.0548,0.4351,0.0843,0.0563,0.0977,-0.0678,-0.0064,0.0380,1,-0.0485
4,1990-05-31,10057,AMT,ACME CLEVELAND CORP,Manufacturing,21,Mach,1098.0,17.9161,1.1521,...,0.0444,0.4351,0.0843,0.0563,0.0977,-0.0678,0.0431,0.0444,0,-0.0205


## 3. Get Charactersitics and Classifiers From Dictionary

In [22]:
determined_characteristics = []
predetermined_characteristics = []
determined_classifications = []

panel_dict = pd.read_csv("FINANCE384_stock_month_data_dictionary.csv")
print(f"Rows           : {len(panel_dict):,}")
panel_dict.head()

determined_characteristics = panel_dict.loc[
    panel_dict["role"] == "characteristic, contemporaneous with t",
    "variable"
].tolist()

predetermined_characteristics = panel_dict.loc[
    panel_dict["role"] == "characteristic, predetermined at t-1",
    "variable"
].tolist()

determined_classifications = panel_dict.loc[
    panel_dict["role"] == "classification, known at t; permitted as a predictor if encoded",
    "variable"
].tolist()

print(f"Rows: {len(panel_dict):,}")
print(f"Determined characteristics: {determined_characteristics}")
print(f"Predetermined characteristics: {predetermined_characteristics}")
print(f"Determined classifications: {determined_classifications}")

Rows           : 27
Rows: 27
Determined characteristics: ['size', 'bm', 'vol12', 'beta60', 'ivol60', 'turnover', 'dollar_volume', 'amihud_illiq', 'divyield', 'mkt_12m', 'mkt_vol_12m', 'down_market']
Predetermined characteristics: ['mom12_2', 'gross_profit', 'roe', 'asset_growth', 'leverage', 'accruals']
Determined classifications: ['industry', 'ff49_code', 'ff49_name']


## 4. Format Pooled data

In [23]:
TARGET = "target_ret_excess_tp1"

panel["month"] = panel["date"].dt.to_period("M")

# Predetermined characteristics: t-1 → t
predetermined_shifted = (
    panel[["permno", "month"] + predetermined_characteristics]
    .assign(month=lambda df: df["month"] + 1)
)

# Return: t → t-1, so it becomes the t+1 target
next_month_return = (
    panel[["permno", "month", "ret_excess_t"]]
    .rename(columns={"ret_excess_t": TARGET})
    .assign(month=lambda df: df["month"] - 1)
)

# Merge t+1 return onto t row
panel = panel.merge(
    next_month_return,
    on=["permno", "month"],
    how="left",
    validate="one_to_one"
)

# Merge predetermined t-1 information onto t row
panel = panel.merge(
    predetermined_shifted,
    on=["permno", "month"],
    how="left",
    validate="one_to_one"
)

# Diagnostics
final_month = (
    panel.groupby("permno")["month"]
    .transform("max")
    .eq(panel["month"])
)

no_target = panel[TARGET].isna()

print(f"Rows                 : {len(panel):,}")
print(f"Usable targets       : {panel[TARGET].notna().sum():,}")
print(f"No month t+1 in data : {no_target.sum():,}")
print(f"  ... stock's last month: {(no_target & final_month).sum():,}")
print(f"  ... gap in coverage   : {(no_target & ~final_month).sum():,}")

panel.loc[:4, ["permno", "month", "ret_excess_t", TARGET]]


Rows                 : 198,298
Usable targets       : 197,016
No month t+1 in data : 1,282
  ... stock's last month: 1,252
  ... gap in coverage   : 30


,permno,month,ret_excess_t,target_ret_excess_tp1
0,10057,1990-01,-0.0540,-0.0179
1,10057,1990-02,-0.0179,-0.0558
2,10057,1990-03,-0.0558,-0.0485
3,10057,1990-04,-0.0485,-0.0205
4,10057,1990-05,-0.0205,-0.1035
